# Notebook 08: Project Integration

**Time:** 35 minutes  
**Prerequisites:** Notebooks 02-07 complete  
**Goal:** Apply this week's tools to your capstone project and plan your data pipeline

This notebook will:
1. Review what you learned this week
2. Ask Claude to design a domain-specific data pipeline for your project
3. Run a mini pipeline on your project's data
4. Generate a project update document

In [1]:
import os, sys, time, importlib
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

import src.llm_client, src.cost_tracker, src.utils, src.config
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("Setup complete -- ready for Notebook 08")

✓ Gemini client initialized (gemini-3-flash-preview)
Setup complete -- ready for Notebook 08


---

## Part 1: Week 3 Review

This week you learned:
- **Web scraping**: trafilatura (traditional) vs Crawl4AI (modern LLM-ready)
- **OCR**: Tesseract (baseline) -> Surya (layout-aware) -> Marker/Docling (2025)
- **ASR**: Whisper, faster-whisper, model size trade-offs
- **Data cleaning**: MinHash dedup, PII removal, DataTrove/FineWeb architecture
- **TTS**: edge-tts (free cloud), Kokoro (local, highest quality)
- **Voice agents**: ASR->LLM->TTS pipeline, Pipecat framework

In [2]:
# Load previous project definition (from HW1/HW2) or create a blank
prev_project = os.path.join(outputs_dir, 'my_project_update.md')
hw2_project = os.path.join(parent_dir, '..', 'Homework2-Submission', 'outputs', 'my_project_update.md')
hw1_project = os.path.join(parent_dir, '..', 'Homework1-Submission', 'outputs', 'my_project_definition.md')

project_context = ""
for path in [hw2_project, hw1_project]:
    if os.path.exists(path):
        with open(path, 'r') as f:
            project_context = f.read()
        print(f"Loaded project context from: {path}")
        print(f"Content preview: {project_context[:300]}...")
        break

if not project_context:
    print("No previous project definition found.")
    print("You'll define your project focus below.")

Loaded project context from: /home/chris/Homework3-Submission/../Homework2-Submission/outputs/my_project_update.md
Content preview: # Project Update — Week 2

**Generated:** 2026-05-02 15:15:11
**Student Name:** Chris

---

## Original Project Definition (Week 1)

# My Research Agent Project
**Created:** 2026-04-07 17:24:31


# MY RESEARCH AGENT PROJECT

## 1. PROJECT TITLE
GitHub Insight Agent – README-Based Repository Summariz...


---

## Part 2: Data Pipeline Strategy

In [5]:
# TODO 1: Define your project's data strategy

print("=" * 65)
print("TODO 1: Project Data Pipeline Design")
print("=" * 65)
print()

project_description = "[My project is about using LLM to translate words and expressions into English, French, and Chinese with definitions, examples, synonyms, antonyms and audio pronunciations. Also, I want to build a voice agent that can answer questions about the words and expressions. Besides, it can extract vocabularies from videos like Youtube ,from PDFs and audio files. Additionally, I want to build a TTS system that can read the definitions and examples aloud. It can also do the dictation of the saved words for users. Finally, I want to build a web interface for users to interact with the system.]"
project_domain = "NLP and language learning"

strategy_prompt = f"""I'm building this capstone project:
{project_description}

Domain: {project_domain}

Previous context:
{project_context[:500] if project_context else 'No previous context'}

This week I learned about:
- Web scraping (trafilatura, Crawl4AI)
- OCR (Tesseract, Marker, Docling)
- ASR (faster-whisper)
- Data cleaning (MinHash dedup, PII removal, quality filtering)
- TTS (edge-tts, Kokoro)
- Voice agents (Pipecat framework)

Design a data pipeline strategy for my project:
1. What data sources should I collect from? (web, PDFs, audio?)
2. Which extraction tools are best for my domain?
3. What cleaning steps are critical for my use case?
4. Should I incorporate voice capabilities? If so, how?
5. What's the estimated data volume and processing time?

Be specific and practical."""

start = time.time()
response = client.generate(
    prompt=strategy_prompt,
    system="You are a senior ML engineer helping design a data pipeline for a capstone project.",
    max_tokens=5700,
    temperature=0.5
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"Response in {elapsed:.1f}s")
    print(format_response(response, verbose=True))
else:
    print(f"Error: {response['error']}")


TODO 1: Project Data Pipeline Design

⚡ Generating with Gemini...
Response in 15.8s
Model: gemini-3-flash-preview
Tokens: 473 in, 1236 out
Stop reason: FinishReason.STOP
This is an ambitious and well-scoped capstone project. As a Senior ML Engineer, I’ve designed a data pipeline strategy that balances **high-quality linguistic extraction** with the **real-time requirements** of a voice agent.

Here is the recommended pipeline strategy for your Language Learning Agent.

---

### 1. Data Sources: Where to collect from?
To build a robust vocabulary extractor and translator, you need two types of data: **Reference Data** (the "truth") and **User Input Data** (the "source").

*   **Reference Data (Linguistic Knowledge):**
    *   **Wiktionary (via dumps):** Excellent for multi-language definitions, synonyms, and etymology.
    *   **WordNet:** Great for structured relationships (hypernyms/antonyms).
    *   **Tatoeba:** A massive database of sentences and translations (perfect for your "exa

In [6]:

todo1_reflection = """


- Which data sources are most relevant for your project?

I found that Extraction Tools: Best-in-Class for my domain  are useful for my project. 
Besides, Stop-word & Frequency Filtering also useful to remove common words and low-frequency terms can help focus on important vocabulary for language learning.

- Which tools from this week will you actually use in your capstone?

 The tools of this week are all useful for my project. For example, I can use TTS (edge-tts, Kokoro) to read the definitions and examples aloud. I can also use Voice agents (Pipecat framework) to build a voice agent that can answer questions about the words and expressions. Additionally, I can use Web scraping (trafilatura, Crawl4AI) to extract vocabularies from videos like Youtube ,from PDFs and audio files. Finally, I can use OCR (Tesseract, Marker, Docling) to extract text from PDFs and images.

- What's the most challenging data quality issue you expect to face?
The most challenging data quality issue I expect to save the words I consulted into a database and make sure the data is clean and organized. This includes removing duplicates, ensuring consistent formatting, and handling any errors that may occur during the extraction process. Additionally, I need to ensure that the audio pronunciations are clear and accurate, which may require some manual review and correction. Finally, I need to make sure that the definitions and examples are relevant and useful for language learners, which may require some curation and filtering of the data.

"""

print()
print(todo1_reflection)





- Which data sources are most relevant for your project?

I found that Extraction Tools: Best-in-Class for my domain  are useful for my project. 
Besides, Stop-word & Frequency Filtering also useful to remove common words and low-frequency terms can help focus on important vocabulary for language learning.

- Which tools from this week will you actually use in your capstone?

 The tools of this week are all useful for my project. For example, I can use TTS (edge-tts, Kokoro) to read the definitions and examples aloud. I can also use Voice agents (Pipecat framework) to build a voice agent that can answer questions about the words and expressions. Additionally, I can use Web scraping (trafilatura, Crawl4AI) to extract vocabularies from videos like Youtube ,from PDFs and audio files. Finally, I can use OCR (Tesseract, Marker, Docling) to extract text from PDFs and images.

- What's the most challenging data quality issue you expect to face?
The most challenging data quality issue I ex

---

## Part 3: Mini Pipeline Demo

In [7]:
# TODO 2: Run a mini data pipeline for your project
#
# Use at least 2 tools from this week to collect and clean
# a small sample of data relevant to your project.

print("=" * 65)
print("TODO 2: Mini Pipeline for Your Project")
print("=" * 65)
print()

# Example: scrape papers + clean them
from src.scraping_utils import scrape_arxiv_abstracts
from src.data_pipeline import run_cleaning_pipeline

# Step 1: Collect data
my_topic = "detect language"
papers = scrape_arxiv_abstracts(topic=my_topic, max_results=5)

# Step 2: Clean the data
raw_texts = [p['abstract'] for p in papers]
pipeline_result = run_cleaning_pipeline(
    texts=raw_texts,
    target_lang="en",
    save_path=os.path.join(outputs_dir, 'my_project_data.json'),
)

print(f"\nCollected and cleaned {len(pipeline_result['cleaned_texts'])} documents for your project.")


TODO 2: Mini Pipeline for Your Project

Scraping arXiv: 'detect language' (max 5 papers)

  [1] AlbumFill: Album-Guided Reasoning and Retrieval for Personalized Image Completio...
      Authors: Yu-Ju Tsai, Brian Price, Qing Liu...
      Abstract: Personalized image completion aims to restore occluded regions in personal photos while preserving identity and appearan...

  [2] SpecKV: Adaptive Speculative Decoding with Compression-Aware Gamma Selection...
      Authors: Shikhar Shukla
      Abstract: Speculative decoding accelerates large language model (LLM) inference by using a small draft model to propose candidate ...

  [3] Unsupervised Machine Learning for Detecting Structural Anomalies in European Reg...
      Authors: Bogdan Oancea
      Abstract: Ensuring the coherence of regional socio-economic statistics is a central task for national statistical institutes. Trad...

  [4] MolmoAct2: Action Reasoning Models for Real-world Deployment...
      Authors: Haoquan Fang, Jiafei Duan

In [8]:

todo2_reflection = """

- What data did you collect and how did you clean it?
In fact, for my project, I don't need to collect data from the web. 
Instead, I will use the LLM to generate the data I need for my project. 
For example, I can use the LLM to generate definitions, examples, synonyms, antonyms and audio pronunciations 
for the words and expressions I want to translate.
 Additionally, I can use the LLM to generate questions and answers for the voice agent that can answer questions about the words and expressions. Finally, I can use the LLM to generate text for the TTS system that can read the definitions and examples aloud.

- Were any documents removed by the pipeline? Why?
No documents were removed by the pipeline. 
However, I did use some cleaning steps to ensure that the data is clean and organized. 
For example, I removed duplicates, ensured consistent formatting, and handled any errors that may occur during the extraction process. Additionally, I made sure that the audio pronunciations are clear and accurate, which may require some manual review and correction. Finally, I made sure that the definitions and examples are relevant and useful for language learners, which may require some curation and filtering of the data.

- How would you scale this to a full dataset for your project?
I will scale this to a full dataset for my project by using the LLM to generate a large amount of data for my project. 


"""

print()
print(todo2_reflection)




- What data did you collect and how did you clean it?
In fact, for my project, I don't need to collect data from the web. 
Instead, I will use the LLM to generate the data I need for my project. 
For example, I can use the LLM to generate definitions, examples, synonyms, antonyms and audio pronunciations 
for the words and expressions I want to translate.
 Additionally, I can use the LLM to generate questions and answers for the voice agent that can answer questions about the words and expressions. Finally, I can use the LLM to generate text for the TTS system that can read the definitions and examples aloud.

- Were any documents removed by the pipeline? Why?
No documents were removed by the pipeline. 
However, I did use some cleaning steps to ensure that the data is clean and organized. 
For example, I removed duplicates, ensured consistent formatting, and handled any errors that may occur during the extraction process. Additionally, I made sure that the audio pronunciations are c

---

## Part 4: Generate Project Update

In [9]:
# Generate project update document
_todo1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[Not completed]'
_todo2 = todo2_reflection.strip() if 'todo2_reflection' in dir() else '[Not completed]'

project_update = f"""# Week 3 Project Update: Pretraining Data & Voice Agents

## Project Description

{project_description}

## Data Pipeline Strategy

{response['content'] if 'error' not in response else 'Error generating strategy'}

## Mini Pipeline Results

- Documents collected: {len(raw_texts) if 'raw_texts' in dir() else 'N/A'}
- Documents after cleaning: {len(pipeline_result['cleaned_texts']) if 'pipeline_result' in dir() else 'N/A'}

## Reflections

### Data Strategy
{_todo1}

### Pipeline Execution
{_todo2}

## Tools I Plan to Use

TTS, Voice Agents, database like postgreSQL etc.

## Next Steps

I may need to build my project with RAG and vector databases, if I find that the data generated by the LLM is not sufficient for my project, especially for the Chinese part of my project.
"""

update_path = os.path.join(outputs_dir, 'my_project_update.md')
with open(update_path, 'w') as f:
    f.write(project_update)

print(f"Project update saved: {update_path}")

Project update saved: ../outputs/my_project_update.md


In [10]:
# Final reflection
full_reflection = f"""
### Data Pipeline Strategy

{_todo1}

---

### Mini Pipeline Results

{_todo2}
"""

reflection_file = append_to_reflection(
    notebook="08",
    section_title="Project Integration",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)

print(f"Reflection saved: {reflection_file}")
print()
tracker.report()

Reflection saved: ../outputs/homework_reflection.md

API COST REPORT
Total API calls:     3
Total input tokens:  1,415
Total output tokens: 3,970
Total cost:          $0.0638

Last 3 calls:
  1. [15:58:26] 3 -- 469in/1353out -- $0.0217
  2. [16:00:07] 3 -- 473in/1381out -- $0.0221
  3. [16:04:49] 3 -- 473in/1236out -- $0.0200


---

## Week 3 Complete!

### Submission Checklist

- [ ] All notebooks (00-08) executed with TODOs filled in
- [ ] `outputs/homework_reflection.md` -- your main deliverable (70%)
- [ ] `outputs/my_project_update.md` -- project integration (20%)
- [ ] `outputs/arxiv_papers.json` -- scraped paper data
- [ ] `outputs/cleaned_data.json` -- cleaned pipeline output
- [ ] Audio files in outputs/ -- TTS demonstrations

### What's Next?

**Week 4:** Retrieval-Augmented Generation (RAG) -- combining LLMs with external knowledge, vector databases, and LangChain.